# 9.5 [실습] 예외 처리와 병렬 실행

## 환경 설정 (Colab)

In [1]:
!pip install langchain_openai==1.1.12 langchain_community==0.4.1 langchain==1.2.14 sqlalchemy numexpr pydantic tenacity nest_asyncio
!pip install -U duckduckgo_search==7.5.1 yfinance ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.2
    Uninstalling langgraph-sdk-0.4.2:
    

In [2]:
import os, getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"]=userdata.get("OPENAI_API_KEY")
except Exception:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass.getpass("OpenAI API Key: ")

OpenAI API Key: ··········


In [3]:
import nest_asyncio; nest_asyncio.apply()

In [4]:
import os
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")  # 불필요한 경고 메시지 숨김

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [5]:
# 1. 실제 금융 데이터 API 도구 (Yahoo Finance 연동)
@tool
def get_stock_price(symbol: str) -> str:
    """특정 종목의 실시간 주가 정보를 조회합니다.
    주의: 한국 주식은 반드시 종목코드 뒤에 '.KS'를 붙여야 합니다 (예: 삼성전자 -> 005930.KS).
    """
    try:
        ticker = yf.Ticker(symbol)
        # fast_info를 사용하면 가볍고 빠르게 현재가를 가져옵니다.
        price = ticker.fast_info['last_price']
        return f"{symbol}의 현재 주가는 {price:,.0f}원입니다."
    except Exception as e:
        return f"주가 정보를 가져오는 데 실패했습니다. 심볼을 확인해주세요: {symbol}"

In [6]:
@tool
def get_exchange_rate(currency_pair: str = "USDKRW=X") -> str:
    """환율 정보를 조회합니다. 기본값은 USD/KRW 환율을 의미하는 'USDKRW=X'입니다."""
    try:
        ticker = yf.Ticker(currency_pair)
        rate = ticker.fast_info['last_price']
        return f"현재 {currency_pair} 환율은 {rate:,.2f}원입니다."
    except Exception as e:
        return f"환율 정보를 가져오는 데 실패했습니다: {currency_pair}"

In [7]:
@tool
def summarize_financials(symbol: str) -> str:
    """특정 기업의 재무 요약 정보(매출, 영업이익률 등)를 제공합니다.
    주의: 한국 주식은 종목코드 뒤에 '.KS'를 붙여야 합니다 (예: 삼성전자 -> 005930.KS).
    """
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        revenue = info.get('totalRevenue', 0)
        margins = info.get('operatingMargins', 0) * 100
        # 매출을 보기 쉽게 조 단위로 변환
        if revenue > 0:
            revenue_str = f"{revenue / 1_000_000_000_000:.2f}조 원"
        else:
            revenue_str = "정보 없음"
        return f"{symbol}의 최근 매출은 {revenue_str}이며, 영업이익률은 약 {margins:.2f}%입니다."
    except Exception as e:
        return f"재무 정보를 불러오는 데 실패했습니다: {symbol}"

In [8]:
# 2. 모델 초기화 및 도구 바인딩
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [get_stock_price, get_exchange_rate, summarize_financials]
llm_with_tools = llm.bind_tools(tools)

In [9]:
# 3. 에이전트 실행 로직
user_input = "삼성전자 주가와 현재 환율, 그리고 삼성전자의 재무 상태를 요약해줘."
ai_msg = llm_with_tools.invoke(user_input)

## 9.5.1 도구 호출 거부와 예외 처리

### 유효성 검사 레이어 구축

In [10]:
from pydantic import BaseModel, Field, field_validator

In [11]:
class StockQuerySchema(BaseModel):
    symbol: str = Field(..., description="종목 코드")

    @field_validator("symbol")
    @classmethod
    def validate_symbol(cls, v: str):
        if not v.isupper():
            raise ValueError("종목 코드는 반드시 대문자여야 합니다.")
        return v

### Tenacity를 이용한 재시도 메커니즘

In [12]:
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(
    stop=stop_after_attempt(3),                      # 최대 3번까지 재시도
    wait=wait_exponential(multiplier=1, min=4, max=10),  # 대기 시간을 지수적으로 증가
    reraise=True                                     # 3번 모두 실패하면 원래 예외 발생
)
def call_external_api(query):
    response = requests.get(f"https://api.finance.com/v1/{query}")
    response.raise_for_status()
    return response.json()

### 자율적 오류 수정(Self-Correction) 루프

### 세 가지 기법의 통합

In [13]:
import requests
from pydantic import BaseModel, Field, field_validator
from tenacity import retry, stop_after_attempt, wait_exponential
from langchain_classic.tools import StructuredTool

In [14]:
class StockQuerySchema(BaseModel):
    symbol: str = Field(..., description="종목 코드 (반드시 대문자, 예: AAPL, 005930.KS)")

    @field_validator("symbol")
    @classmethod
    def validate_symbol(cls, v: str):
        if not v.isupper():
            raise ValueError(f"종목 코드 '{v}'는 반드시 대문자여야 합니다. 예: AAPL, 005930.KS")
        return v

In [15]:
@retry(
    stop=stop_after_attempt(3),                      # 최대 3번 재시도
    wait=wait_exponential(multiplier=1, min=4, max=10),  # 대기 간격을 지수적으로 늘리되 4~10초로 제한
    reraise=True                                     # 3번 모두 실패 시 원래 예외 발생
)
def _fetch_stock_price_from_api(symbol: str) -> dict:
    """실제 외부 API를 호출하는 내부 함수. 에러 시 자동 재시도합니다."""
    response = requests.get(
        f"https://api.finance.com/v1/quote?symbol={symbol}", timeout=5
    )
    response.raise_for_status()
    return response.json()

In [16]:
def get_stock_price_safe(symbol: str) -> str:
    """Pydantic 검증 + Tenacity 재시도가 적용된 주가 조회 함수"""
    try:
        # 1단계: Pydantic으로 입력값 검증
        validated = StockQuerySchema(symbol=symbol)
        # 2단계: Tenacity 재시도 보호 하에 API 호출
        data = _fetch_stock_price_from_api(validated.symbol)
        price = data.get("price", "알 수 없음")
        return f"{validated.symbol}의 현재 주가는 {price}원입니다."
    except ValueError as e:
        # Pydantic 검증 실패: LLM이 잘못된 형식으로 인자를 넘긴 경우
        # 이 메시지가 Observation으로 에이전트에 전달되어 Self-Correction을 트리거합니다.
        return f"입력값 오류: {e} 종목 코드를 대문자로 다시 입력해주세요."
    except Exception as e:
        # API 호출 최종 실패 (3번 재시도 후에도 실패)
        return f"API 호출 실패: {e}. 잠시 후 다시 시도하거나 다른 종목을 조회해주세요."

# AgentExecutor에 등록할 때는 StructuredTool로 래핑
safe_stock_tool = StructuredTool(
    name="get_stock_price_safe",
    func=get_stock_price_safe,
    description="특정 종목의 현재 주가를 조회합니다. 종목 코드는 반드시 대문자로 입력하세요 (예: AAPL, 005930.KS).",
    args_schema=StockQuerySchema
)

## 9.5.2 병렬 실행 (asyncio)

In [17]:
import asyncio

selected_tool = {
    "get_stock_price": get_stock_price,
    "get_exchange_rate": get_exchange_rate,
    "summarize_financials": summarize_financials,
}

async def call_one(tool_call):
    tool = selected_tool[tool_call["name"].lower()]
    # yfinance 호출은 동기(blocking) 함수이므로, 별도 스레드에서 실행해
    # 이벤트 루프를 막지 않도록 asyncio.to_thread로 감쌉니다.
    return await asyncio.to_thread(tool.invoke, tool_call["args"])

async def run_parallel(ai_msg):
    # gather가 모든 호출을 동시에 시작하고, 전부 끝날 때까지 기다립니다.
    return await asyncio.gather(*[call_one(tc) for tc in ai_msg.tool_calls])

results = await run_parallel(ai_msg)  # 노트북에서는 nest_asyncio 덕분에 await를 바로 사용
for r in results:
    print(r)

005930.KS의 현재 주가는 318,000원입니다.
현재 USDKRW=X 환율은 1,532.24원입니다.
005930.KS의 최근 매출은 388.34조 원이며, 영업이익률은 약 42.75%입니다.
